# Session 6 notebook: are characteristics covariances?

**Question.** Kelly, Pruitt and Su (2019, "KPS") let each stock's factor betas depend on its characteristics, estimate a few latent factors, and find that once betas move with characteristics, the characteristics leave no alpha. Their title states the conclusion: *characteristics are covariances*. Kozak and Nagel (JFE, "KN") show when such factors can span the SDF at all, and that under those conditions IPCA reduces to plain PCA on characteristic portfolios. You will estimate IPCA by hand, then check KN's shortcut on the same data.

| Part | What you do | When |
|---|---|---|
| 1 | Build the managed portfolios IPCA works with | before class, 10 min |
| 2 | Estimate IPCA by alternating least squares; fit for 1 to 6 factors | in class |
| 3 | Add an "anomaly" intercept: do characteristics still predict returns beyond betas? | in class |
| 4 | Kozak-Nagel: IPCA as PCA on OLS factors, and why it needs Monday's ridge | in class |
| 5 | Out of sample: IPCA against Monday's shrunk SDF | in class |
| 6 | Discuss | in class |

**Data:** the same `week3_data.zip` as Session 5 (link on the course notes page).

## Part 0. Setup (before class)

In [ ]:
import numpy as np, pandas as pd, time
import matplotlib.pyplot as plt
from pathlib import Path
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.4f}".format)

# The class data (week3_data.zip). The link is on the course notes page, not in this public notebook.
DATA_ZIP_LINK = ""     # <- paste the Google Drive link to week3_data.zip between the quotes, then run this cell

import re, zipfile, urllib.request
DATA_DIR = Path("data")
if not (DATA_DIR / "w3_panel.parquet").exists():
    fid = re.search(r"/d/([\w-]+)|[?&]id=([\w-]+)", DATA_ZIP_LINK)
    assert fid, "Paste the week3_data.zip link from the course notes page into DATA_ZIP_LINK above, then rerun this cell."
    fid = fid.group(1) or fid.group(2)
    urllib.request.urlretrieve(f"https://drive.usercontent.google.com/download?id={fid}&export=download&confirm=t", "week3_data.zip")
    assert zipfile.is_zipfile("week3_data.zip"), "The download did not return a zip. Check that the link is shared with you, and try again."
    zipfile.ZipFile("week3_data.zip").extractall(".")          # creates ./data/
    print("downloaded and unzipped week3_data.zip")

panel = pd.read_parquet(DATA_DIR / "w3_panel.parquet")
ff = pd.read_parquet(DATA_DIR / "ff_factors_monthly.parquet").set_index("month")
signals = pd.read_csv(DATA_DIR / "w3_signals.csv")

STYLES = ["size", "value", "momentum", "beta", "resid_vol", "leverage", "liquidity", "profitability"]
CHARS = STYLES + signals["acronym"].tolist()
GROUP = {**{s: "barra" for s in STYLES}, **dict(zip(signals["acronym"], signals["group"]))}

# KPS's scaling: each month, rank every characteristic across stocks and map it to [-0.5, 0.5].
# The 52 Chen-Zimmermann characteristics already come this way; rank the 8 Barra styles too.
panel[STYLES] = panel.groupby("month")[STYLES].rank(pct=True).sub(0.5).astype("float32")
months = sorted(panel.month.unique())
print(f"{len(panel):,} stock-months, {len(months)} months, {len(CHARS)} characteristics")

## Part 1. Managed portfolios (before class)

IPCA models each stock's return as

> r<sub>i,t+1</sub> = z<sub>i,t</sub>&prime; &Gamma; f<sub>t+1</sub> + &epsilon;<sub>i,t+1</sub>

z<sub>i,t</sub> is the stock's vector of characteristics (plus a constant), so its **betas are z<sub>i,t</sub>&prime;&Gamma;**: they move when its characteristics move. &Gamma; (L characteristics &times; K factors) is the same for every stock and every month. f are K latent factors.

Everything IPCA needs from the stock-level data is in two objects per month:

- x<sub>t+1</sub> = Z<sub>t</sub>&prime; r<sub>t+1</sub> / N<sub>t</sub>: the returns of L **managed portfolios**, one per characteristic, each stock weighted by its characteristic. These are Session 5's characteristic portfolios.
- W<sub>t</sub> = Z<sub>t</sub>&prime; Z<sub>t</sub> / N<sub>t</sub>: the cross-sectional second-moment matrix of the characteristics.

In [ ]:
Z, R = {}, {}
for m, g in panel.groupby("month"):
    Z[m] = np.column_stack([np.ones(len(g)), g[CHARS].values.astype(float)])     # first column: constant
    R[m] = g["y"].values
NAMES = ["const"] + CHARS
L = len(NAMES)
X = np.array([Z[m].T @ R[m] / len(R[m]) for m in months])        # T x L managed-portfolio returns
W = np.array([Z[m].T @ Z[m] / len(R[m]) for m in months])        # T x L x L
T = len(months)
print("managed portfolios:", X.shape, "  W:", W.shape)
print("smallest eigenvalue of the average W:", f"{np.linalg.eigvalsh(W.mean(0)).min():.1e}", "(remember Monday)")

**Check yourself.** The first managed portfolio uses the constant, so it is the equal-weighted market. What is the managed portfolio for `Mom12m`? Why is the smallest eigenvalue of W so close to zero?

## Part 2. Estimate IPCA (in class)

KPS minimize the sum of squared errors over &Gamma; and all the f<sub>t</sub>. There is no closed form, but each half is a regression (KPS eqs. 6 and 7):

1. **Given &Gamma;, find f<sub>t</sub>:** each month, regress the stocks' returns on their betas Z<sub>t</sub>&Gamma;. That is a cross-sectional regression, like Barra, with K estimated "styles": f<sub>t</sub> = (&Gamma;&prime;W<sub>t</sub>&Gamma;)<sup>&minus;1</sup>&Gamma;&prime;x<sub>t</sub>.
2. **Given the f<sub>t</sub>, find &Gamma;:** one pooled regression of returns on characteristics &times; factors.

Alternate until nothing changes. Then rotate so that &Gamma;&prime;&Gamma; = I and the factors are uncorrelated, ordered by variance (the identification KPS use).

In [ ]:
def ipca(X, W, K, alpha=False, iters=1000, tol=1e-8):
    """IPCA by alternating least squares, on managed portfolios X (T x L) and W (T x L x L).
    alpha=True adds an anomaly intercept: returns get an extra z'Gamma_alpha that is not a beta times a factor."""
    T, L = X.shape
    G = np.linalg.eigh(X.T @ X / T)[1][:, ::-1][:, :K]          # start from plain PCA of the managed portfolios
    Ga = np.zeros(L)
    for it in range(iters):
        F = np.array([np.linalg.solve(G.T @ W[t] @ G, G.T @ (X[t] - W[t] @ Ga)) for t in range(T)])   # step 1
        Fa = np.column_stack([F, np.ones(T)]) if alpha else F
        A = sum(np.kron(W[t], np.outer(Fa[t], Fa[t])) for t in range(T))                        # step 2
        b = sum(np.kron(X[t], Fa[t]) for t in range(T))
        Gn = np.linalg.solve(A, b).reshape(L, Fa.shape[1])
        Gn, Ga_n = (Gn[:, :K], Gn[:, K]) if alpha else (Gn, Ga)
        change = np.abs(Gn - G).max() + np.abs(Ga_n - Ga).max()
        G, Ga = Gn, Ga_n
        if change < tol:
            break
    F = np.array([np.linalg.solve(G.T @ W[t] @ G, G.T @ (X[t] - W[t] @ Ga)) for t in range(T)])
    # identification: Gamma'Gamma = I, factors uncorrelated and ordered by variance, positive means
    U = np.linalg.cholesky(G.T @ G).T
    G, F = G @ np.linalg.inv(U), F @ U.T
    _, V = np.linalg.eigh(np.atleast_2d(np.cov(F.T)))
    V = V[:, ::-1]
    G, F = G @ V, F @ V
    s = np.where(F.mean(0) < 0, -1, 1)
    return G * s, F * s, Ga, it + 1

def r2(G, F, Ga=None, idx=None):
    """KPS's total R2 (fitted with realized factors) and predictive R2 (fitted with the factors' average), stock level."""
    idx = range(T) if idx is None else idx
    lam = F.mean(0)
    ga = np.zeros(L) if Ga is None else Ga
    e_tot = e_pred = tot = 0.0
    for t in idx:
        beta = Z[months[t]] @ G
        r = R[months[t]]
        e_tot += ((r - Z[months[t]] @ ga - beta @ F[t]) ** 2).sum()
        e_pred += ((r - Z[months[t]] @ ga - beta @ lam) ** 2).sum()
        tot += (r ** 2).sum()
    return 1 - e_tot / tot, 1 - e_pred / tot

In [ ]:
fits, rows = {}, []
for K in range(1, 7):
    t0 = time.time()
    G, F, _, it = ipca(X, W, K)
    fits[K] = (G, F)
    tot, pred = r2(G, F)
    rows.append({"K": K, "iterations": it, "total R2 %": 100 * tot, "predictive R2 %": 100 * pred, "seconds": time.time() - t0})
pd.DataFrame(rows).set_index("K").round(3)

**Read it.** *Total R&sup2;* asks how well betas times realized factors describe returns (risk). *Predictive R&sup2;* asks how well betas times the factors' *average* describe returns (expected returns). KPS, with 36 characteristics, 1962-2014: about 18.6% and 0.7% with five factors. A predictive R&sup2; that looks tiny is normal for monthly stock returns: Session 2's note on R&sup2; vs. the Sharpe ratio shows what 0.5% of predictable variance is worth.

In [ ]:
K = 5                                     # Your turn: try 3
G, F = fits[K]
loadings = pd.DataFrame(G, index=NAMES, columns=[f"factor {k + 1}" for k in range(K)])
for k in range(K):
    col = loadings.iloc[:, k]
    top = col.reindex(col.abs().sort_values(ascending=False).index).head(5)
    print(f"factor {k + 1} (SR {F[:, k].mean() / F[:, k].std() * np.sqrt(12):.2f}):  " +
          ", ".join(f"{n} {v:+.2f}" for n, v in top.items()))

**Interpret.** Each factor's betas are a combination of characteristics; the five largest are listed. Try to name the factors. You will find it hard: near-duplicates such as `momentum` and `Mom12m`, or `DolVol` and `Illiquidity`, often enter with *opposite* signs. That is Monday's collinearity again. KPS's identification (&Gamma;&prime;&Gamma; = I, ordered factors) pins down a rotation; it does not stop the model from trading off twins. KPS find only about 10 of their 36 characteristics matter. Which groups seem to matter here?

## Part 3. Is there an anomaly left? (in class)

KPS's test: add an intercept that also depends on characteristics, &alpha;<sub>i,t</sub> = z<sub>i,t</sub>&prime;&Gamma;<sub>&alpha;</sub>. If characteristics are only proxies for betas, &Gamma;<sub>&alpha;</sub> should be zero. KPS reject zero alpha with one factor but not with five or more, and conclude that "characteristics are covariances".

In [ ]:
rows = []
for K in [1, 3, 5]:
    Gu, Fu, Ga, it = ipca(X, W, K, alpha=True)
    tot_u, pred_u = r2(Gu, Fu, Ga)
    rows.append({"K": K, "predictive R2 % (no alpha)": 100 * r2(*fits[K])[1], "predictive R2 % (with alpha)": 100 * pred_u,
                 "W_alpha = Gamma_a'Gamma_a": Ga @ Ga,
                 "largest alpha characteristics": ", ".join(f"{NAMES[i]} {Ga[i]:+.3f}" for i in np.argsort(-np.abs(Ga))[:4])})
alpha_table = pd.DataFrame(rows).set_index("K")
alpha_table

**Read it.** Look at which characteristics carry the largest alpha coefficients, and at their signs. Are they near-duplicates from Monday's correlation table, with offsetting signs? If so, the alpha is the same problem as Monday's OLS Barra: the model pays one characteristic and charges its twin.

KPS judge W<sub>&alpha;</sub> with a bootstrap: simulate returns from the zero-alpha model, re-estimate, and see how often W<sub>&alpha;</sub> is as large as the real one. It takes several minutes, so it is off by default.

In [ ]:
RUN_BOOTSTRAP = False      # Your turn (optional): set True. About 15 s per draw with K = 5; B = 30 takes ~8 minutes.
B, K_TEST = 30, 5
if RUN_BOOTSTRAP:
    Gu, Fu, Ga, _ = ipca(X, W, K_TEST, alpha=True)
    Wa = Ga @ Ga
    D = np.array([X[t] - W[t] @ (Ga + Gu @ Fu[t]) for t in range(T)])          # managed-portfolio residuals
    rng = np.random.default_rng(0)
    draws = []
    for b in range(B):
        q = rng.standard_t(5, T) / np.sqrt(5 / 3)                               # wild bootstrap, as in KPS
        Xb = np.array([W[t] @ (Gu @ Fu[t]) + q[t] * D[rng.integers(T)] for t in range(T)])   # null: no alpha
        Ga_b = ipca(Xb, W, K_TEST, alpha=True)[2]
        draws.append(Ga_b @ Ga_b)
    print(f"W_alpha = {Wa:.4f}; bootstrap p-value = {np.mean(np.array(draws) >= Wa):.2f} ({B} draws)")

## Part 4. Kozak and Nagel: the shortcut, and what it needs (in class)

KN (Example 5, eq. 32-33) show that *if* the covariance matrix of returns has the structure under which characteristic portfolios span the SDF, IPCA needs no iteration:

1. Form the **OLS factors**: each month, regress returns on all characteristics (Barra without industries). The slopes are O<sub>t</sub> = W<sub>t</sub><sup>&minus;1</sup> x<sub>t</sub>.
2. Take the top K principal components Q of their covariance matrix.
3. The factors are f<sub>t</sub> = (Q&prime;W<sub>t</sub>Q)<sup>&minus;1</sup>Q&prime;x<sub>t</sub>, with &Gamma; = Q.

Try it, and compare with Part 2.

In [ ]:
def kn_shortcut(K, lam=0.0, idx=None):
    """Kozak-Nagel's PCA-on-OLS-factors version of IPCA. lam > 0 uses Monday's ridge in the OLS step."""
    idx = range(T) if idx is None else idx
    O = np.array([np.linalg.solve(W[t] + lam * np.eye(L), X[t]) for t in idx])      # (ridge) OLS factors
    Q = np.linalg.eigh(np.cov(O.T))[1][:, ::-1][:, :K]
    F = np.array([np.linalg.solve(Q.T @ W[t] @ Q, Q.T @ X[t]) for t in range(T)])
    return Q, F

def same_space(G1, G2):
    """Smallest cosine of the principal angles between two K-dimensional spaces: 1 = identical, 0 = some direction is missed."""
    q1, q2 = np.linalg.qr(G1)[0], np.linalg.qr(G2)[0]
    return np.linalg.svd(q1.T @ q2, compute_uv=False).min()

rows = []
for K in [1, 3, 5]:
    G, F = fits[K]
    rows.append({"K": K, "IPCA total R2 %": 100 * r2(G, F)[0], "IPCA pred R2 %": 100 * r2(G, F)[1]})
    for lam, tag in [(0.0, "KN shortcut"), (0.03, "KN + ridge 0.03")]:
        Q, Fq = kn_shortcut(K, lam)
        tot, pred = r2(Q, Fq)
        rows[-1].update({f"{tag} total R2 %": 100 * tot, f"{tag} pred R2 %": 100 * pred,
                         f"{tag}: same space as IPCA": same_space(G, Q)})
pd.DataFrame(rows).set_index("K").round(3).T

**Read it.** The plain shortcut does badly, especially with one factor. The reason is Monday's: W<sub>t</sub> has a nearly zero eigenvalue, so the OLS factors W<sub>t</sub><sup>&minus;1</sup>x<sub>t</sub> are dominated by huge, offsetting bets on near-duplicate characteristics, and PCA picks up *that* variance. A little ridge in the OLS step brings the shortcut close to IPCA's fit (the factors are close, not identical). KN's equivalence is a statement about population moments; in a sample with collinear characteristics it needs shrinkage to show up.

**Your turn.** Change the ridge value (0.001, 0.01, 0.1) and K. Where does the shortcut come closest to IPCA?

### 4.2 The bigger point in KN

KN's first conclusion (p. 37): horse races between characteristics and factor models "have no economic content other than exposing the shortcomings of heuristic factor construction," and "do not lead to insights about competing economic theories of risk premia and mispricing." KPS say much the same about their own factors (their Section 2): a factor is just a source of comovement, and it could come from correlated *mispricing*. And Monday's KNS argument cuts both ways: even a behavioral mispricing has to line up with high-variance factors, or arbitrageurs would trade it away. So zero alpha after IPCA says characteristics line up with covariances. It does not say the premium is compensation for risk.

## Part 5. Out of sample (in class)

Estimate on 2015-2020, test on 2021-2026. For IPCA, &Gamma; is fixed from the training period; each test month's factors come from that month's characteristics and returns (step 1), and the factor weights are the training-period tangency portfolio. For comparison, Monday's shrunk SDF on the same characteristics, with &gamma; = 0.1 &times; trace(&Sigma;) taken from Monday's table (which peeked at the test period; Monday's cross-validated value gives a similar answer).

In [ ]:
train = np.array([m < "2021-01" for m in months])
def sharpe(r): return r.mean() / r.std() * np.sqrt(12)

rows = []
for K in range(1, 7):
    G, _, _, _ = ipca(X[train], W[train], K)
    F = np.array([np.linalg.solve(G.T @ W[t] @ G, G.T @ X[t]) for t in range(T)])
    w = np.linalg.solve(np.atleast_2d(np.cov(F[train].T)), F[train].mean(0))
    rows.append({"model": f"IPCA, K = {K}", "SR in-sample": sharpe(F[train] @ w), "SR out-of-sample": sharpe(F[~train] @ w)})

# Monday's ridge SDF on the managed portfolios (market-neutralized, gamma = 0.1 x trace)
mkt = ff.loc[months, "mktrf"].values
Xc = X[:, 1:] - np.outer(mkt, [np.polyfit(mkt, X[:, j], 1)[0] for j in range(1, L)])
mu, S = Xc[train].mean(0), np.cov(Xc[train].T)
for g in [0.0, 0.1]:
    b = np.linalg.solve(S + g * np.trace(S) * np.eye(L - 1), mu)
    rows.append({"model": f"Session 5 SDF, gamma = {g} x trace", "SR in-sample": sharpe(Xc[train] @ b), "SR out-of-sample": sharpe(Xc[~train] @ b)})
pd.DataFrame(rows).set_index("model").round(2)

**Read it.** How many IPCA factors do you need before the out-of-sample Sharpe ratio settles? How does a five-factor IPCA model compare with 60 characteristics shrunk with ridge? Then use Monday's rule: with five years of test data, what Sharpe ratio would you need for t = 2? Is the gap between any two rows here something you could detect?

## Part 6. Discuss (in class)

1. IPCA's betas move with characteristics; Session 3's Barra model *assumed* exposures equal characteristics. What does IPCA add, and what does it cost?
2. KPS conclude that characteristics are covariances, not anomalies. KN say horse races like Part 3 cannot separate risk from mispricing. Who is right, and what evidence would settle it?
3. Kent Daniel's discussion asks what quants actually do: max w&prime;X&phi; &minus; &lambda;w&prime;&Sigma;w &minus; transaction costs. Where do IPCA and Monday's ridge fit into that problem? Which piece is the "expected return model" and which is the "risk model"?
4. Everything here ignores trading costs and uses equal-weighted portfolios with small stocks. Which result would you expect to shrink most once costs are in? (Session 8.)
5. Recall Case #1. If a value factor showed up as an IPCA factor with zero alpha, would that prove a mean-reversion (mispricing) view of value wrong?